#### All Imports

In [693]:
import os
from langgraph.graph import StateGraph, END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing_extensions import TypedDict, Literal, Annotated
from langchain.messages import HumanMessage, SystemMessage, AnyMessage
from langgraph.graph.message import add_messages
from langchain.tools import tool
from tavily import TavilyClient
from pydantic import BaseModel, Field
from pprint import pprint
from IPython.display import Markdown, display, Image
from langgraph.prebuilt import ToolNode

In [694]:
load_dotenv(".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [695]:
basic_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)
advanced_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)

In [696]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [697]:
# Prompts - Read Content from Files
f = open("./prompts/generations.md")
GENERATION_PROMPT = f.read()
f = open("./prompts/optimisation.md")
OPTIMISATION_PROMPT = f.read()
f = open("./prompts/scoring.md")
SCORING_PROMPT = f.read()
f = open("./prompts/validation.md")
VALIDATION_PROMPT = f.read()

In [698]:
# Output of scroing agent
class Score(BaseModel):
    score: int | None
    confidence: float
    sufficient_data: bool
    missing_information: list[str]
    description: str

# Output of generation offre
class Offre(BaseModel):
    offre: str = Field(
        description="""
            Return either:
                - A personalized marketing offer.
                - A clear human action message explaining that more customer information is required.
        """
    )

# Output of validation offre
class IsValid(BaseModel):
    is_valid: bool = Field(
        description="Boolean value indicating whether the offer valid or invalid."
    )
    description: str = Field(
        description="A clear and detailed explanation of why the offer is valid or invalid."
    )

class MarketingState(TypedDict):
    user_input: str
    customer_data: str
    score: Score
    offer: Offre
    valid: IsValid
    optimized_version: Offre
    next: Literal["INIT", "FETCHING", "SCORING", "GENERATION", "VALIDATION", "OPTMISATION", "END"]
    messages: Annotated[list[AnyMessage], add_messages]

#### Fetcher Agent

In [699]:
# Fetcher Agent
"""
    In Backend We Should Handle This Part
        --> User Enter Prompt
        --> User Select CSV, Excel
        --> Use Use MCP 
"""

# This agent can use tools to connect to DB, Read execel files, ...
def fetcher_agent(state: MarketingState):
    """
        Decide where is the data: (Database tool, File tool, )
    """
    client_data = """
        I have a client who spent 10 dollars on online shopping products during the last month.
        The client made 3 purchases, mainly buying electronics and accessories.
        Their average order value is around 3.33 dollars, and they usually purchase during weekends.
        The client has been active for 6 months and shows moderate engagement with marketing campaigns.
    """
    return {"client_data": str(client_data), "next": "SCORING"}


#### Scoring Agent

In [700]:
# Tools for Scoring Agent
@tool
def getBusinessInformation():
    """
        Get business Informations For Customer Scoring
    """
    print("TOOL CALL: getBusinessInformation")
    # should get file that contain (Brand, Contact to Use, Ton, ...)
    return "No business Informations"


llm_tools = basic_llm.bind_tools([getBusinessInformation])
llm_score = basic_llm.with_structured_output(Score)

def scoring_agent(state: MarketingState):
    customer_data = state.get("customer_data")
    messages = state.get("messages", [])

    if not messages:
        messages = [
            SystemMessage(content=SCORING_PROMPT),
            HumanMessage(
                content=f"""
                    Based on this customer data, score the client: \n\n 
                        Customer Data = {customer_data} \n\n
                """
            ),
        ]

    # First LLM: tool calling
    response = llm_tools.invoke(messages)

    # If tools are needed, let ToolNode execute tools
    if response.tool_calls:
        return {
            "messages": [response]
        }

    # Second LLM: structured output
    score = llm_score.invoke(messages + [response])

    return {
        "messages": [response],
        "score": score,
        "next": "GENERATION",
    }


scoring_tools = ToolNode(tools=[getBusinessInformation])

def should_continue(state: MarketingState):
    print("GO TO: ")
    last_message = state.get("messages", [])[-1]
    print("last_message: ", last_message)
    if last_message.tool_calls:
        print("scoring_tools")
        return "scoring_tools"
    else:
        print("END")
        return END


In [701]:
workflow = StateGraph(MarketingState)
workflow.add_node("scoring_agent", scoring_agent)
workflow.add_node("scoring_tools", scoring_tools)
workflow.add_edge("scoring_tools", "scoring_agent")
workflow.set_entry_point("scoring_agent")
workflow.add_conditional_edges(
    "scoring_agent",
    should_continue, 
    {
        "scoring_tools": "scoring_tools",
        END: END
    }
)

app = workflow.compile()
# display(Image(app.get_graph().draw_mermaid_png()))

init_input = {
    "user_input": "",
    "customer_data": "no data available yet",
    "score": None,
    "messages": []
}

result = app.invoke(input=init_input)


GO TO: 
last_message:  content='' additional_kwargs={'tool_calls': [{'id': 'mqjw5mrbd', 'function': {'arguments': '{}', 'name': 'getBusinessInformation'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 415, 'total_tokens': 424, 'completion_time': 0.013603547, 'completion_tokens_details': None, 'prompt_time': 0.02928152, 'prompt_tokens_details': None, 'queue_time': 0.01784375, 'total_time': 0.042885067}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_8639719ff2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fc40b-fbd0-7592-9a4c-c04ccc50c496-0' tool_calls=[{'name': 'getBusinessInformation', 'args': {}, 'id': 'mqjw5mrbd', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 415, 'output_tokens': 9, 'total_tokens': 424}
scoring_tools
TOOL CALL: getBusinessInformation


BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: attempted to call tool 'brave_search' which was not in request.tools", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=brave_search>{"query": "get business information for customer scoring"}</function>'}}

#### Generation Agent

In [ ]:
# Tools
@tool
def getExternalInformation() -> str:
    """
        Retrieves external business information.
        Args:
            This tool does not require any arguments.
    """
    return """
        Business Type: Coffe Shope
    """

@tool("HowToWriteMarketingOffre")
def HowToWriteMarketingOffre(query: str, limit: int) -> list:
    """
        Search On Internet Using Of How to Write Marketing Offre
        Args:
            query: Search term to look for
            limit: Maximum number of results to return
    """
    response = tavily_client.search(query=query, max_results=limit)
    results = response.get("results", [])
    content = []
    for res in results:
        content.append({"url": res.get("url", ""), "content": res.get("content", "")})
    return content

# Generation Agent
def generation_agent(state: MarketingState):
    customer_data = state.get("client_data", "No Client Data")
    customer_score = state.get("score", "0")
    user_input = state.get('user_input', "Generate Marketing Offre based on the customer data")
    messages = [
        SystemMessage(content=GENERATION_PROMPT),
        HumanMessage(
            content=f"""
            Generate marketing offer.

            User Request:
            {user_input}

            Customer Data:
            {customer_data}
            
            Customer Score:
            {customer_score}/100
            """
        )
    ]
    
    llm_with_tools = basic_llm.bind_tools([HowToWriteMarketingOffre, getExternalInformation])
    tool_response = llm_with_tools.invoke(messages)
    print(tool_response)
    final_messages = [*messages, tool_response]
    response = llm_with_tools.with_structured_output(Offre).invoke(final_messages)
    return {
        "offre": response.offre,
        "next": "VALIDATION"
    }


In [ ]:
# Test Generation Agent
client_data = fetcher_agent({})["client_data"]
print(client_data)
state: MarketingState = {
    "user_input": "Generate Marketing Offre based on the customer data",
    "client_data": client_data,
    "offre": None,
    "score": 0,
}
result = generation_agent(state)



        I have a client who spent 10 dollars on online shopping products during the last month.
        The client made 3 purchases, mainly buying electronics and accessories.
        Their average order value is around 3.33 dollars, and they usually purchase during weekends.
        The client has been active for 6 months and shows moderate engagement with marketing campaigns.
    
content='' additional_kwargs={'tool_calls': [{'id': 'syre66hxw', 'function': {'arguments': '{}', 'name': 'getExternalInformation'}, 'type': 'function'}, {'id': 'pjvnfqgh7', 'function': {'arguments': '{"limit":5,"query":"persuasive marketing offer for low-spending customer with moderate engagement"}', 'name': 'HowToWriteMarketingOffre'}, 'type': 'function'}, {'id': 'vscyafg9k', 'function': {'arguments': '{"limit":5,"query":"personalized marketing offer for customer with average order value $3.33"}', 'name': 'HowToWriteMarketingOffre'}, 'type': 'function'}, {'id': 'wqj7z4nta', 'function': {'arguments': '{"li

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=Offre> {"offre": "Get 10% off on your next purchase of electronics and accessories. As a valued customer, we noticed you\'ve been shopping with us for 6 months and have shown moderate engagement with our marketing campaigns. To encourage your continued loyalty, we\'re offering you an exclusive discount. Simply use the code WEEKEND10 at checkout to redeem your discount. Don\'t miss out on this amazing opportunity to save on your favorite products!"}'}}

In [ ]:
display(Markdown(result.get("offre")))

Get 10% off on your next purchase of electronics and accessories. As a valued customer, we noticed you've been shopping with us for 6 months and have shown moderate engagement with our marketing campaigns. To encourage your continued loyalty, we're offering you this exclusive discount. Simply use the code WEEKEND10 at checkout to redeem your offer. Don't miss out on this amazing opportunity to save on your favorite products!

#### Validation Agent

In [ ]:
# Tools
@tool
def getValidationRules():
    """
        Get Rules To Validate The Offre
    """
    
    # Read File From Data Called validation.md
    return "no business rules"

# Validation Agent
def validation_agent(state: MarketingState):
    # Get Offre
    offre = state.get("offre", "no offre")
    messages = [SystemMessage(content=VALIDATION_PROMPT), HumanMessage(content=
        f"""
        Offre: \n\n
            {offre}
        """
    )]
    
    response = basic_llm.with_structured_output(IsValid).invoke(messages)
    print(type(response.is_valid))
    print(response.is_valid)
    print(response.description)
    if response.is_valid:
        return {"offre": response, "next": "END"}
    return {"offre": response, "next": "OPTMISATION"}

In [ ]:
# Test Validation Agent
offre = """Bonjour Ahmed,
Nous sommes ravis de vous proposer une offre spéciale en votre nom !
Vous avez déjà dépensé 209,70€ chez nous
Nous avons remarqué que vous avez acheté des produits de qualité tels que des parfums, des chaussures et des t-shirts. Nous sommes convaincus que vous allez adorer notre nouvelle collection de produits de luxe !
Découvrez nos meilleures offres
Parfum exclusif : 50% de réduction sur notre nouveau parfum, disponible uniquement pendant une semaine !
Chaussures de luxe : 30% de réduction sur nos chaussures de luxe, conçues pour les pieds les plus exigeants !
T-shirt premium : 20% de réduction sur nos t-shirts premium, faits avec les matériaux les plus confortables !
N'oubliez pas de profiter de nos offres spéciales
Livraison gratuite : sur tous les commandes supérieures à 100€
Retour gratuit : sur tous les produits achetés chez nous
Cliquez ici pour découvrir nos offres [lien vers la page d'offres]
Nous sommes impatients de vous voir chez nous !
Cordialement, L'équipe de [votre nom de l'entreprise]
"""

# Test Generation Agent
state: MarketingState = {
    "offre": offre,
}
result = validation_agent(state)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=IsValid> {"is_valid": false, "description": "L\\\'offre est considérée comme non valide car elle ne respecte pas les règles de base de validation. En effet, l\\\'offre est conditionnelle et nécessite un achat minimum de 100€ pour bénéficier de la livraison gratuite."}</function>'}}

#### Optimisation Offre

In [ ]:
# Tools for Optimisation offre


# Agent for offre Optmisation
def optmisation_agent(state: MarketingState):
    # Get Offre
    offre = state.get("offre", "None")
    if (type(state.get("valid")) == "IsValid"):
        feedback = state.get("valid").description
    else:
        feedback = "None"
    messages = [SystemMessage(content=OPTIMISATION_PROMPT), HumanMessage(content=f"""
        Based on This feedback: \n\n
            {feedback}
        Optimise this Offre: \n\n 
            {offre}
        """)]
    
    response = basic_llm.with_structured_output(Offre).invoke(messages)
    return {"offre": response, "next": "END"}

In [ ]:
# Test Validation Agent
offre = """Bonjour Ahmed,
Nous sommes ravis de vous proposer une offre spéciale en votre nom !
Vous avez déjà dépensé 209,70€ chez nous
Nous avons remarqué que vous avez acheté des produits de qualité tels que des parfums, des chaussures et des t-shirts. Nous sommes convaincus que vous allez adorer notre nouvelle collection de produits de luxe !
Découvrez nos meilleures offres
Parfum exclusif : 50% de réduction sur notre nouveau parfum, disponible uniquement pendant une semaine !
Chaussures de luxe : 30% de réduction sur nos chaussures de luxe, conçues pour les pieds les plus exigeants !
T-shirt premium : 20% de réduction sur nos t-shirts premium, faits avec les matériaux les plus confortables !
N'oubliez pas de profiter de nos offres spéciales
Livraison gratuite : sur tous les commandes supérieures à 100€
Retour gratuit : sur tous les produits achetés chez nous
Cliquez ici pour découvrir nos offres [lien vers la page d'offres]
Nous sommes impatients de vous voir chez nous !
Cordialement, L'équipe de [votre nom de l'entreprise]
"""

# Test Generation Agent
client_data = fetcher_agent({})["client_data"]
state: MarketingState = {
    "user_input": "",
    "client_data": client_data,
    "marketing_offer": "",
    "score": 85,
    "offre": offre,
    "next": "SCORING",
    "valid": None
}
result = optmisation_agent(state)
display(Markdown(result.get('offre').offre))

Bonjour Ahmed,
Nous sommes ravis de vous proposer une offre spéciale en votre nom !
Vous avez déjà dépensé 209,70€ chez nous
Nous avons remarqué que vous avez acheté des produits de qualité tels que des parfums, des chaussures et des t-shirts. Nous sommes convaincus que vous allez adorer notre nouvelle collection de produits de luxe !
Découvrez nos meilleures offres
Parfum exclusif : 50% de réduction sur notre nouveau parfum, disponible uniquement pendant une semaine !
Chaussures de luxe : 30% de réduction sur nos chaussures de luxe, conçues pour les pieds les plus exigeants !
T-shirt premium : 20% de réduction sur nos t-shirts premium, faits avec les matériaux les plus confortables !
N'oubliez pas de profiter de nos offres spéciales
Livraison gratuite : sur tous les commandes supérieures à 100€
Retour gratuit : sur tous les produits achetés chez nous
Cliquez ici pour découvrir nos offres [lien vers la page d'offres]
Nous sommes impatients de vous voir chez nous !
Cordialement, L'équipe de [votre nom de l'entreprise]

#### Build graph

In [ ]:
# Routing
def router(state: MarketingState):
    return state["next"]

In [ ]:
workflow = StateGraph(MarketingState)

In [ ]:
workflow.add_node("fetcher_agent", fetcher_agent)
workflow.add_node("scoring_agent", scoring_agent)
# workflow.add_node("generation_agent", generation_agent)
# workflow.add_node("validation_agent", validation_agent)
# workflow.add_node("optmisation_agent", optmisation_agent)

workflow.set_entry_point(fetcher_agent)



